In [1]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [2]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [3]:
import pandas as pd
import numpy as np

In [4]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA

In [5]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [6]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


## EXAMPLES OF BYTE COMPARISONS

In [7]:
w1 = 'abhor'
w2 = 'cleft'
w3 = 'frown'
w1b = byte_encode_words(w1)
w2b = byte_encode_words(w2)
w3b = byte_encode_words(w3)

In [8]:
# no letters in common
w1b & w2b

0

In [9]:
# letters in common
w1b & w3b

147456

In [10]:
# bitwise or
w1b | w2b

673975

In [11]:
# this is the same as directly above
byte_encode_words('abhorcleft')

673975

# BUILD LEVEL 2

In [16]:
l2_list = np.full(shape = (10000000, 3), fill_value = -1, dtype = np.int32)
row_index = 0
for w1_be, w2_be in combinations(word_byte_list, 2):
    if w1_be & w2_be == 0:   
        # they share no letters in common
        l2 = w1_be | w2_be                          
        
        l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
        row_index += 1

# trim the data frame
l2_list = l2_list[:row_index, :]
print(l2_list.shape)
l2_df = pd.DataFrame(data = l2_list, columns = ['w1b', 'w2b', 'l2'])

(3213696, 3)


In [17]:
l2_df['l2'].unique().shape

(640023,)

In [ ]:
tl2_df = l2_df.drop_duplicates(subset = 'l2').reset_index(drop = True)

In [21]:
tl2_array = tl2_df['l2'].to_numpy(dtype = np.int32)

In [84]:
# two arrays

In [ ]:
# this is a long array...
tl2_array.shape

(640023,)

In [87]:
tl2t = tl2_array[:100,]

In [94]:
tl2t

array([  284959,   553247,   808991,   815135,  1134879,   160031,
       17453087, 16930847,   807199,  1855519, 17324063,   282815,
         151967,    21919,    55455,    28895,   675999,   807071,
       18895007,  4477087, 20992159,    20927,    22943,    29087,
          29135,    30095,   291215,   553359,   315791,   152975,
         153999,   184719,   676239,  1200527,   807311,  2118047,
        6312335, 18895247,   414095,  1593503,  1331407,  1069727,
        1077455,  1078415,  1202319,  1208463,  1331359, 16799903,
          31007,   291103,   413983,   676127,  2642207,   284735,
         546879,   579615,   154655,  1072159,  4479007,   547119,
          31055,    32015,   555279,   317711,   579855,   547103,
        2119967,  1333279,  1079375,  1080335, 16832543,   708639,
         938015,  4608031, 16929055,   153887,   161039,   446735,
        1202207,  1724447,  1209359,  1732623,  1462303,  1462415,
        1986575,  1724559,  4870287, 17485839, 17715215,  1331

In [85]:
word_byte_array.shape[0]

5977

In [89]:
n_wba = word_byte_array.shape[0]

In [ ]:
# I want each value to be repeated 5977 times across. In the x direction
r_tl2t = np.reshape(np.repeat(a = tl2t, repeats = n_wba), shape = (100, n_wba))

In [98]:
r_tl2t

array([[ 284959,  284959,  284959, ...,  284959,  284959,  284959],
       [ 553247,  553247,  553247, ...,  553247,  553247,  553247],
       [ 808991,  808991,  808991, ...,  808991,  808991,  808991],
       ...,
       [1077535, 1077535, 1077535, ..., 1077535, 1077535, 1077535],
       [1601807, 1601807, 1601807, ..., 1601807, 1601807, 1601807],
       [1863695, 1863695, 1863695, ..., 1863695, 1863695, 1863695]],
      shape=(100, 5977), dtype=int32)

In [102]:
(r_tl2t[:, 1] == r_tl2t[:, 100]).all()

np.True_

In [118]:
r_wba = np.reshape(word_byte_array, shape = (1, n_wba))

In [119]:
r_wba = np.repeat(a = r_wba, repeats = 100, axis = 0)

In [120]:
r_wba

array([[   20491,     8219,   786451, ..., 50597904, 50336004, 50344192],
       [   20491,     8219,   786451, ..., 50597904, 50336004, 50344192],
       [   20491,     8219,   786451, ..., 50597904, 50336004, 50344192],
       ...,
       [   20491,     8219,   786451, ..., 50597904, 50336004, 50344192],
       [   20491,     8219,   786451, ..., 50597904, 50336004, 50344192],
       [   20491,     8219,   786451, ..., 50597904, 50336004, 50344192]],
      shape=(100, 5977), dtype=int32)

In [110]:
r_wba[0, :]

array([ 20491,  20491,  20491, ..., 131101, 131101, 131101],
      shape=(5977,), dtype=int32)

In [121]:
(r_wba[0, :] == word_byte_array).all()

np.True_

In [122]:
# here we go!!!!!
outcome = r_tl2t & r_wba

In [ ]:
outcome

In [72]:
tl2_test = tl2_array[:100]

In [73]:
tl2_test.shape

(100,)

In [75]:
rep_tl2_test = np.reshape(np.repeat(a = tl2_test, repeats = word_byte_array.shape[0]), shape = (100, word_byte_array.shape[0]))

In [76]:
rep_tl2_test

array([[ 284959,  284959,  284959, ...,  284959,  284959,  284959],
       [ 553247,  553247,  553247, ...,  553247,  553247,  553247],
       [ 808991,  808991,  808991, ...,  808991,  808991,  808991],
       ...,
       [1077535, 1077535, 1077535, ..., 1077535, 1077535, 1077535],
       [1601807, 1601807, 1601807, ..., 1601807, 1601807, 1601807],
       [1863695, 1863695, 1863695, ..., 1863695, 1863695, 1863695]],
      shape=(100, 5977), dtype=int32)

In [77]:
(rep_tl2_test[:, 0] ==  rep_tl2_test[:, 99]).all()

np.True_

In [ ]:
rwba[:, 1]

(100,)

In [81]:
(rwba[:, 0] == rwba[:, 100]).all()

np.False_

In [45]:
# testo = tl2_array & word_byte_array
testo = tl2_test | outcome

ValueError: operands could not be broadcast together with shapes (100,) (100,5977) 

In [47]:
tl2_test[:10] | outcome[:10, :]

ValueError: operands could not be broadcast together with shapes (10,) (10,5977) 

In [ ]:
np.bitwise_or(x1 = )

# BUILD LEVELS 3 THROUGH 5

In [ ]:
# so, now, let's try computing all possible pairs
total_output = np.full(shape = (1000000, 9), fill_value= -1, dtype = np.int32)
row_index = 0
for i_row, row in l2_df.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # indexer for l3
    positional_idx_l3 = (word_byte_array & l2) == 0

    # l3 words with different letters
    output_array_w3b = word_byte_array[positional_idx_l3]    

    # l3 accumulated letters
    output_array_l3 = output_array_w3b | l2

    ## enumerate level 3
    for w3b, l3 in zip(output_array_w3b, output_array_l3):

        # build level 4

        # l4 idx
        positional_idx_l4 = (word_byte_array & l3) == 0

        # words with different letters
        output_array_w4b = word_byte_array[positional_idx_l4]    
        
        # accumulated letters
        output_array_l4 = output_array_w4b | l3

        ## enumerate level 5
        for w4b, l4 in zip(output_array_w4b, output_array_l4):

            # build level 5

            # l5 idx
            positional_idx_l5 = (word_byte_array & l4) == 0
            
            # words with different letters
            output_array_w5b = word_byte_array[positional_idx_l5]    

            if output_array_w5b.size > 0:
                    
                # accumulated letters
                output_array_l5 = output_array_w5b | l4

                ## gather and combine the output
                for w5b, l5 in zip(output_array_w5b, output_array_l5):

                    temp_list = np.array([w1b, w2b, w3b, w4b, w5b, l2, l3, l4, l5], dtype = np.int32)
                    total_output[row_index, :] = temp_list                   
                    row_index += 1


    if i_row % 10000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row, row_index)
    


# CREATE AND SAVE OUTPUT

In [ ]:
total_output = total_output[:row_index, :]
col_names = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b', 'l2', 'l3', 'l4', 'l5']
l5_df = pd.DataFrame(data = total_output, columns = col_names)


In [ ]:
l5_df.shape

In [ ]:
l5_df.head()

In [ ]:
l5_df.tail()

In [ ]:
l5_df.to_csv(path_or_buf='l5.txt', sep = '\t', index = False)
